# 05. Демо дедупликатора — от DuckDB до дерева SKU

Этот notebook показывает end-to-end механику дедупликатора на маленьком срезе реального куба. Мы берём похожие SKU из DuckDB, прогоняем их через bi-encoder, FAISS и cross-encoder, затем собираем группы и показываем результат как дерево.

**Результат:** понятная витрина `Level 1 -> Level 2`: сверху каноничный SKU группы, ниже реальные SKU, которые модель предлагает объединить.

## Оглавление

- [Введение: что здесь считается результатом](#введение-что-здесь-считается-результатом)
- [Ход действий](#ход-действий)
- [0. Локальные настройки](#0-локальные-настройки)
- [1. Подготовка окружения](#1-подготовка-окружения)
- [2. Подключаемся к DuckDB и берём category-run](#2-подключаемся-к-duckdb-и-берём-category-run)
- [3. Собираем SKU-каталог из куба](#3-собираем-sku-каталог-из-куба)
- [4. Быстрый поиск похожих кандидатов](#4-быстрый-поиск-похожих-кандидатов)
- [5. Выбираем маленький demo-pool SKU](#5-выбираем-маленький-demo-pool-sku)
- [6. Bi-encoder: строим embeddings](#6-bi-encoder-строим-embeddings)
- [7. FAISS: ищем ближайшие SKU по embeddings](#7-faiss-ищем-ближайшие-sku-по-embeddings)
- [8. Cross-encoder: пересчитываем score пары](#8-cross-encoder-пересчитываем-score-пары)
- [9. Собираем SKU в группы](#9-собираем-sku-в-группы)
- [10. Join с кубом и выбор каноничного SKU](#10-join-с-кубом-и-выбор-каноничного-sku)
- [11. Дерево: Level 1 -> Level 2](#11-дерево-level-1---level-2)
- [12. Сохраняем demo-export](#12-сохраняем-demo-export)
- [13. Итоговые выводы и как пользоваться](#13-итоговые-выводы-и-как-пользоваться)

## Ход действий

1. Подключаемся к DuckDB и берём маленький, но реальный category-run.
2. Собираем SKU-каталог, где один ряд = один товар на marketplace.
3. Находим похожие пары быстрым поиском, затем уточняем их bi-encoder, FAISS и cross-encoder.
4. Строим группы SKU и выбираем каноничный товар для верхнего уровня дерева.
5. Показываем результат как `Level 1 -> Level 2` и сохраняем demo-export.

Важно: это интерактивное demo, а не полный production run. Объём специально ограничен, чтобы на защите можно было быстро показать механику и примеры склейки.


## Введение: что здесь считается результатом

- **Кандидаты** — пары SKU, найденные быстрым лексическим поиском внутри выбранной категории.
- **Bi-encoder** — модель превращает каждый SKU в dense-вектор.
- **FAISS** — быстро ищет ближайших соседей по этим векторам.
- **Cross-encoder** — смотрит на пару SKU целиком и пересчитывает финальный score.
- **Группа** — связная компонента по positive edges, то есть по парам, которые модель считает дублями.
- **Каноничный SKU** — представитель группы: сначала побеждает средний model score, затем продажи.
- **Level 1** — каноничный SKU и агрегаты всей группы.
- **Level 2** — реальные SKU, которые входят в группу.


## 0. Локальные настройки

Обычно достаточно поменять `MY_CATEGORY_RUN`, `MY_DUCKDB_PATH` и лимиты demo-среза. Секреты, cache моделей и API-ключи остаются в `.env` / окружении.


In [ ]:
# === MY notebook settings ===
MY_CATEGORY_RUN = "coconut_oil"  # sauces | coconut_oil | soap
MY_DUCKDB_PATH = None  # None = MPSTATS_DUCKDB_PATH, ./mpstats.duckdb или /Users/exoldoff/Desktop/mpstats/mpstats.duckdb
# Лимиты demo-среза: это не полный прогон куба.
MY_MIN_SALES_QTY = 15
MY_MAX_SKUS_FROM_DUCKDB = 1200
MY_LEXICAL_MIN_SIMILARITY = 0.50
MY_LEXICAL_CANDIDATE_LIMIT = 350
MY_LEXICAL_PAIR_SCORING_LIMIT = 120_000
MY_DEMO_SEED_PAIR_COUNT = 120
MY_DEMO_SKU_LIMIT = 180

# Модельный прогон.
MY_BI_ENCODER_MODEL = "embedding_e5_small"
MY_FAISS_TOP_K = 10
MY_FAISS_MIN_SCORE = 0.78
MY_BI_ENCODER_THRESHOLD = 0.86
MY_CROSS_ENCODER_MODEL = "ft_bge_reranker_v2_m3"
MY_CROSS_ENCODER_THRESHOLD = None  # None = threshold из registry/fine-tuned config; можно поставить свой float
MY_CROSS_ENCODER_PAIR_LIMIT = 300
MY_SKIP_CROSS_ENCODER = False
MY_REQUIRE_CROSS_ENCODER = False

# Локальные fine-tuned cross-encoder модели, которых нет в общем model_registry.
# Если backup лежит в другом месте, поменяйте только model_path.
MY_FINE_TUNED_CROSS_ENCODER_MODELS = {
    "ft_bge_reranker_v2_m3": {
        "display_name": "BGE v2 m3 marketplace fine-tune",
        "model_path": "/Users/exoldoff/Desktop/mpstats_server_backup_20260630_041745/artifacts/models/dedup/exoldoff/bge-reranker-v2-m3-cross-encoder-marketplaces-rus/final",
        "hf_model_id": "exoldoff/bge-reranker-v2-m3-cross-encoder-marketplaces-rus",
        "method_name": "ft_bge_reranker_v2_m3",
        "batch_size": 32,
        "activation": "sigmoid",
        "threshold": 0.911269,  # dev threshold_cost_sensitive: меньше false-merge, чем recall-heavy пороги
    },
}

# Группировка и вывод.
MY_GROUPING_ALGORITHM = "connected_components"
MY_EXPORT_DEMO_CSV = True
MY_DISPLAY_ROWS = 25
MY_RANDOM_STATE = 42

## 1. Подготовка окружения

Подключаем numpy/pandas, FAISS и research helpers. FAISS импортируется до загрузки transformer-моделей: на локальном macOS это помогает избежать native crash при смешанном импорте FAISS и torch.


In [ ]:
from __future__ import annotations

import math
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import faiss  # import before sentence-transformers model loading to avoid local FAISS/torch segfault
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 220)



PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "research" / "dedup").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.dedup import (
    CandidateGenerationConfig,
    ComponentConfig,
    GraphGroupingConfig,
    build_graph_groups,
    generate_candidate_pairs,
    prepare_product_records,
    resolve_category_run,
    resolve_embedding_model_spec,
    resolve_model_spec,
    resolve_run_paths,
)
from research.dedup.matchers.cross_encoder import CrossEncoderConfig, CrossEncoderMatcher
from research.dedup.model_registry import CROSS_ENCODER_BACKEND, ModelManager, model_text_prefix

PROJECT_ROOT


def _resolve_notebook_path(raw_path: object) -> Path | None:
    '''Превращает notebook-путь или env-expanded строку в абсолютный Path.'''
    if raw_path is None or str(raw_path).strip() == "":
        return None
    path = Path(os.path.expandvars(str(raw_path))).expanduser()
    if not path.is_absolute():
        path = PROJECT_ROOT / path
    return path.resolve()


def _resolve_demo_cross_encoder(alias_or_name: str) -> dict[str, object]:
    '''Возвращает runtime-конфиг demo cross-encoder из локального fine-tuned блока или registry.'''
    fine_tuned_cfg = MY_FINE_TUNED_CROSS_ENCODER_MODELS.get(alias_or_name)
    if fine_tuned_cfg is not None:
        local_path = _resolve_notebook_path(fine_tuned_cfg.get("model_path"))
        hf_model_id = str(fine_tuned_cfg.get("hf_model_id") or "").strip()
        if local_path is not None and local_path.exists():
            model_name = str(local_path)
            source_note = "local_path"
        elif hf_model_id:
            model_name = hf_model_id
            source_note = "hf_model_id"
        else:
            expected = str(local_path) if local_path is not None else "<empty model_path>"
            raise FileNotFoundError(
                f"Fine-tuned model {alias_or_name!r} не найден: {expected}. "
                "Проверьте MY_FINE_TUNED_CROSS_ENCODER_MODELS['model_path']."
            )
        return {
            "display_name": str(fine_tuned_cfg.get("display_name") or alias_or_name),
            "model_name": model_name,
            "method_name": str(fine_tuned_cfg.get("method_name") or alias_or_name),
            "batch_size": int(fine_tuned_cfg.get("batch_size") or 16),
            "device": fine_tuned_cfg.get("device"),
            "trust_remote_code": bool(fine_tuned_cfg.get("trust_remote_code") or False),
            "prompts": fine_tuned_cfg.get("prompts"),
            "default_prompt_name": fine_tuned_cfg.get("default_prompt_name"),
            "activation": fine_tuned_cfg.get("activation"),
            "threshold": fine_tuned_cfg.get("threshold"),
            "source_note": source_note,
        }

    spec = resolve_model_spec(alias_or_name, backend=CROSS_ENCODER_BACKEND)
    return {
        "display_name": spec.alias,
        "model_name": spec.model_name,
        "method_name": spec.method_name or alias_or_name,
        "batch_size": spec.batch_size or 16,
        "device": spec.device,
        "trust_remote_code": spec.trust_remote_code,
        "prompts": spec.prompts,
        "default_prompt_name": spec.default_prompt_name,
        "activation": None,
        "threshold": spec.fusion_threshold_high,
        "source_note": "model_registry",
    }


## 2. Подключаемся к DuckDB и берём category-run

Открываем локальный куб только на чтение и сразу фильтруем нужный project/category. Таблица фиксированная — `mpstats_products`, потому что это основной контракт данных после pipeline/classification.


In [ ]:
import duckdb

CATEGORY_RUN = resolve_category_run(MY_CATEGORY_RUN)
RUN_PATHS = resolve_run_paths(PROJECT_ROOT, CATEGORY_RUN)
PRODUCTS_TABLE = "mpstats_products"


def _resolve_duckdb_path(raw_path: str | None) -> Path:
    '''Находит существующий DuckDB-куб для read-only demo-загрузки.'''
    candidates = []
    if raw_path:
        candidates.append(Path(raw_path).expanduser())
    env_path = os.environ.get("MPSTATS_DUCKDB_PATH")
    if env_path:
        candidates.append(Path(env_path).expanduser())
    candidates.extend(
        [
            PROJECT_ROOT / "mpstats.duckdb",
            Path("/Users/exoldoff/Desktop/mpstats/mpstats.duckdb"),
        ]
    )
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    checked = ", ".join(str(path) for path in candidates)
    raise FileNotFoundError(f"Не нашёл DuckDB. Укажите MY_DUCKDB_PATH. Проверенные пути: {checked}")


def _quote_ident(name: str) -> str:
    '''Экранирует имя DuckDB-идентификатора двойными кавычками.'''
    return '"' + name.replace('"', '""') + '"'


DUCKDB_PATH = _resolve_duckdb_path(MY_DUCKDB_PATH)
CATEGORY_ALIASES = list(CATEGORY_RUN.category_aliases)
PROJECT_NAME = CATEGORY_RUN.project_name

with duckdb.connect(str(DUCKDB_PATH), read_only=True) as con:
    tables = {row[0] for row in con.execute("SHOW TABLES").fetchall()}
    if PRODUCTS_TABLE not in tables:
        raise RuntimeError(f"В DuckDB нет таблицы {PRODUCTS_TABLE!r}. Доступные таблицы: {sorted(tables)[:20]}")
    table_info = con.execute(f"PRAGMA table_info('{PRODUCTS_TABLE}')").fetchdf()
    available_columns = set(table_info["name"])

    where_clauses = []
    params: list[object] = []
    if "Категория" in available_columns:
        placeholders = ", ".join(["?"] * len(CATEGORY_ALIASES))
        where_clauses.append(f'{_quote_ident("Категория")} IN ({placeholders})')
        params.extend(CATEGORY_ALIASES)
    if PROJECT_NAME:
        if "__project_name" not in available_columns:
            raise RuntimeError("Для category-run нужен project-фильтр, но в кубе нет __project_name.")
        where_clauses.append(f'{_quote_ident("__project_name")} = ?')
        params.append(PROJECT_NAME)
    where_sql = f" WHERE {' AND '.join(where_clauses)}" if where_clauses else ""
    products_raw = con.execute(f"SELECT * FROM {_quote_ident(PRODUCTS_TABLE)}{where_sql}", params).fetchdf()

required_columns = {"Маркетплейс", "Артикул", "SKU"}
missing_required = sorted(required_columns - set(products_raw.columns))
if missing_required:
    raise RuntimeError(f"В срезе DuckDB не хватает обязательных колонок: {missing_required}")

print(f"Category run: {CATEGORY_RUN.slug} — {CATEGORY_RUN.display_name}; project={PROJECT_NAME!r}")
print(f"DuckDB: {DUCKDB_PATH}")
print(f"Rows loaded before sales filter: {len(products_raw):,}")
display(table_info[["name", "type"]])

## 3. Собираем SKU-каталог из куба

Здесь реальные месячные строки куба схлопываются до одного `node_id = marketplace::article`. Продажи и выручка агрегируются, а title/brand/вес берутся из строки с максимальными продажами.

In [ ]:
def _clean_article(value: object) -> str:
    '''Нормализует артикул из DuckDB, убирая хвост .0 после Excel-like чисел.'''
    if pd.isna(value):
        return ""
    text = str(value).strip()
    return text[:-2] if text.endswith(".0") else text


def _norm_marketplace(value: object) -> str:
    '''Приводит marketplace к стабильному ключу для node_id.'''
    if pd.isna(value):
        return "unknown_marketplace"
    text = " ".join(str(value).casefold().strip().split())
    return text or "unknown_marketplace"


def _first_nonempty(values: pd.Series) -> object:
    '''Возвращает первое непустое значение из серии или pd.NA.'''
    clean = values.dropna()
    if clean.empty:
        return pd.NA
    text_clean = clean.astype(str).str.strip()
    text_clean = clean[text_clean.ne("")]
    return text_clean.iloc[0] if not text_clean.empty else clean.iloc[0]


def _unique_join(values: pd.Series, limit: int = 10) -> str:
    '''Собирает короткую строку уникальных значений для audit-колонок.'''
    clean = sorted(set(values.dropna().astype(str).str.strip()) - {""})
    suffix = "" if len(clean) <= limit else f" ... +{len(clean) - limit}"
    return ", ".join(clean[:limit]) + suffix


products = products_raw.copy()
if "Продажи, шт" in products.columns:
    products["Продажи, шт"] = pd.to_numeric(products["Продажи, шт"], errors="coerce").fillna(0)
    rows_before = len(products)
    products = products[products["Продажи, шт"].ge(MY_MIN_SALES_QTY)].copy()
    print(f"Sales filter: {len(products):,} / {rows_before:,} rows kept (Продажи, шт >= {MY_MIN_SALES_QTY})")
else:
    print("В кубе нет 'Продажи, шт': sales filter и sales-агрегаты будут пропущены.")

products["node_id"] = products["Маркетплейс"].map(_norm_marketplace) + "::" + products["Артикул"].map(_clean_article)

sales_col = "Продажи, шт" if "Продажи, шт" in products.columns else None
revenue_col = "Выручка, руб" if "Выручка, руб" in products.columns else None
price_col = "Средняя цена, руб" if "Средняя цена, руб" in products.columns else None
unit_col = "Вес, кг (ед.)" if "Вес, кг (ед.)" in products.columns else None
total_col = "Вес, кг" if "Вес, кг" in products.columns else None
subcategory_col = "Подкатегория" if "Подкатегория" in products.columns else None

rows: list[dict[str, object]] = []
for node_id, group in products.groupby("node_id", sort=False):
    if sales_col:
        representative = group.loc[group[sales_col].fillna(0).idxmax()]
    else:
        representative = group.iloc[0]

    sales_qty = float(group[sales_col].fillna(0).sum()) if sales_col else math.nan
    revenue_rub = float(group[revenue_col].fillna(0).sum()) if revenue_col else math.nan
    avg_price = revenue_rub / sales_qty if sales_qty and math.isfinite(revenue_rub) else math.nan
    unit_amount = representative.get(unit_col) if unit_col else math.nan
    total_amount = representative.get(total_col) if total_col else math.nan
    multipack_count = math.nan
    if pd.notna(unit_amount) and pd.notna(total_amount) and float(unit_amount) > 0:
        multipack_count = max(1, round(float(total_amount) / float(unit_amount)))

    rows.append(
        {
            "node_id": node_id,
            "Маркетплейс": representative.get("Маркетплейс"),
            "Артикул": _clean_article(representative.get("Артикул")),
            "SKU": representative.get("SKU"),
            "Бренд": representative.get("Бренд"),
            "Категория": representative.get("Категория"),
            "Подкатегория": representative.get(subcategory_col) if subcategory_col else "",
            "Вес, кг (ед.)": unit_amount,
            "Вес, кг": total_amount,
            "multipack_count": multipack_count,
            "sales_qty": sales_qty,
            "revenue_rub": revenue_rub,
            "avg_price_rub": avg_price,
            "cube_rows": len(group),
            "cube_months": _unique_join(group["месяц"]) if "месяц" in group.columns else "",
            "cube_projects": _unique_join(group["__project_name"]) if "__project_name" in group.columns else "",
        }
    )

sku_catalog_full = pd.DataFrame(rows).sort_values("sales_qty", ascending=False).reset_index(drop=True)
sku_catalog = sku_catalog_full.head(MY_MAX_SKUS_FROM_DUCKDB).copy()

catalog_summary = pd.DataFrame(
    [
        {"metric": "raw_rows_after_sales_filter", "value": len(products)},
        {"metric": "unique_sku_nodes_full", "value": sku_catalog_full["node_id"].nunique()},
        {"metric": "unique_sku_nodes_in_demo_pool", "value": sku_catalog["node_id"].nunique()},
        {"metric": "demo_pool_sales_qty", "value": round(float(sku_catalog["sales_qty"].fillna(0).sum()), 2)},
        {"metric": "demo_pool_revenue_rub", "value": round(float(sku_catalog["revenue_rub"].fillna(0).sum()), 2)},
    ]
)

display(catalog_summary)
display(sku_catalog.head(MY_DISPLAY_ROWS))

## 4. Быстрый поиск похожих кандидатов

Это предварительный фильтр, а не финальное решение. Он быстро выбирает небольшую группу похожих SKU, чтобы дальше показать работу моделей на осмысленном demo-pool.


In [ ]:
base_candidate_cfg = CandidateGenerationConfig(
    min_similarity=MY_LEXICAL_MIN_SIMILARITY,
    max_candidates=MY_LEXICAL_CANDIDATE_LIMIT,
    max_pair_candidates_for_scoring=MY_LEXICAL_PAIR_SCORING_LIMIT,
    collapse_exact_title_same_brand=False,
    collapse_empty_brand_exact_titles=False,
)

lexical_candidates = generate_candidate_pairs(sku_catalog, base_candidate_cfg)
if lexical_candidates.empty:
    raise RuntimeError(
        "Дешёвый поиск не нашёл кандидатов. Попробуйте увеличить MY_MAX_SKUS_FROM_DUCKDB "
        "или снизить MY_LEXICAL_MIN_SIMILARITY."
    )

candidate_display_cols = [
    "baseline_similarity_score",
    "is_cross_marketplace_pair",
    "is_hard_negative_candidate",
    "marketplace_a",
    "sku_a",
    "brand_a",
    "title_a",
    "marketplace_b",
    "sku_b",
    "brand_b",
    "title_b",
    "unit_amount_a",
    "total_amount_a",
    "unit_amount_b",
    "total_amount_b",
]
candidate_display_cols = [column for column in candidate_display_cols if column in lexical_candidates.columns]

print(f"Lexical candidate pairs: {len(lexical_candidates):,}")
display(lexical_candidates[candidate_display_cols].head(MY_DISPLAY_ROWS))

## 5. Выбираем маленький demo-pool SKU

Берём SKU из верхних лексических пар. Так notebook остаётся быстрым, но в demo-pool уже есть достаточно похожих товаров, чтобы увидеть поведение bi-encoder, FAISS и cross-encoder.


In [ ]:
selected_node_ids: list[str] = []
seen: set[str] = set()
for row in lexical_candidates.head(MY_DEMO_SEED_PAIR_COUNT).itertuples(index=False):
    for node_id in [str(row.raw_record_id_a), str(row.raw_record_id_b)]:
        if node_id not in seen:
            selected_node_ids.append(node_id)
            seen.add(node_id)
        if len(selected_node_ids) >= MY_DEMO_SKU_LIMIT:
            break
    if len(selected_node_ids) >= MY_DEMO_SKU_LIMIT:
        break

record_cfg = CandidateGenerationConfig(
    collapse_exact_title_same_brand=False,
    collapse_empty_brand_exact_titles=False,
)
records_all = prepare_product_records(sku_catalog, record_cfg)
demo_records = records_all[records_all["raw_record_id"].isin(selected_node_ids)].copy()
# Сохраняем порядок выбора из candidate-пар.
order = {node_id: idx for idx, node_id in enumerate(selected_node_ids)}
demo_records["demo_order"] = demo_records["raw_record_id"].map(order)
demo_records = demo_records.sort_values("demo_order").reset_index(drop=True)

if len(demo_records) < 2:
    raise RuntimeError("В demo-pool меньше двух SKU. Увеличьте лимиты в первой ячейке.")

print(f"Demo SKU pool: {len(demo_records):,} SKU")
display(
    demo_records[
        [
            "raw_record_id",
            "marketplace",
            "sku",
            "brand",
            "title",
            "subcategory",
            "unit_amount",
            "total_amount",
            "multipack_count",
        ]
    ].head(MY_DISPLAY_ROWS)
)

## 6. Bi-encoder: строим embeddings

Bi-encoder кодирует каждый SKU отдельно. В demo-тексте оставляем бренд, title и весовые признаки, чтобы модель видела не только название.

In [ ]:
embedding_spec = resolve_embedding_model_spec(MY_BI_ENCODER_MODEL)
manager = ModelManager()
embedding_model = manager.load_embedding_model(MY_BI_ENCODER_MODEL, backend=embedding_spec.backend)
text_prefix = model_text_prefix(MY_BI_ENCODER_MODEL, backend=embedding_spec.backend) or ""


def _format_model_text(row: pd.Series, *, prefix: str = "") -> str:
    '''Собирает структурированный SKU-текст для bi-encoder модели.'''
    parts = []
    brand = str(row.get("brand") or "").strip()
    title = str(row.get("title") or "").strip()
    if brand:
        parts.append(f"бренд: {brand}")
    if title:
        parts.append(f"название: {title}")
    for label, column in [
        ("вес единицы кг", "unit_amount"),
        ("общий вес кг", "total_amount"),
        ("штук в упаковке", "multipack_count"),
    ]:
        value = row.get(column)
        if pd.notna(value):
            parts.append(f"{label}: {value}")
    text = " | ".join(parts)
    return f"{prefix}{text}".strip()


demo_records["model_text"] = demo_records.apply(lambda row: _format_model_text(row, prefix=text_prefix), axis=1)
texts = demo_records["model_text"].tolist()
embeddings = embedding_model.encode(
    texts,
    batch_size=embedding_spec.batch_size or 64,
    normalize_embeddings=True,
    convert_to_numpy=True,
)
embeddings = np.ascontiguousarray(np.asarray(embeddings, dtype="float32"))
if embeddings.ndim != 2 or len(embeddings) != len(demo_records):
    raise RuntimeError(f"Bi-encoder вернул неожиданную форму embeddings: {embeddings.shape}")

print(f"Bi-encoder: {embedding_spec.model_name}; vectors: {embeddings.shape}")
display(demo_records[["raw_record_id", "brand", "title", "model_text"]].head(5))

## 7. FAISS: ищем ближайшие SKU по embeddings

Так выглядит retrieval-слой: FAISS берёт vectors из bi-encoder и отдаёт top-k соседей. Здесь используется `IndexFlatIP`, потому что vectors уже L2-normalized, а inner product равен cosine similarity.

In [ ]:
faiss_top_k = min(MY_FAISS_TOP_K + 1, len(demo_records))
index = faiss.IndexFlatIP(int(embeddings.shape[1]))
index.add(embeddings)
distances, indices = index.search(embeddings, faiss_top_k)

pair_rows: dict[tuple[int, int], dict[str, object]] = {}
for left_idx, (score_row, index_row) in enumerate(zip(distances, indices, strict=False)):
    rank = 0
    for score, right_idx in zip(score_row, index_row, strict=False):
        right_idx = int(right_idx)
        if right_idx < 0 or right_idx == left_idx:
            continue
        if float(score) < MY_FAISS_MIN_SCORE:
            continue
        pair_key = tuple(sorted((left_idx, right_idx)))
        rank += 1
        existing = pair_rows.get(pair_key)
        if existing is None or float(score) > float(existing["bi_encoder_score"]):
            left = demo_records.iloc[pair_key[0]]
            right = demo_records.iloc[pair_key[1]]
            pair_rows[pair_key] = {
                "raw_record_id_a": left["raw_record_id"],
                "raw_record_id_b": right["raw_record_id"],
                "marketplace_a": left["marketplace"],
                "marketplace_b": right["marketplace"],
                "sku_a": left["sku"],
                "sku_b": right["sku"],
                "title_a": left["title"],
                "title_b": right["title"],
                "brand_a": left["brand"],
                "brand_b": right["brand"],
                "unit_amount_a": left["unit_amount"],
                "unit_amount_b": right["unit_amount"],
                "total_amount_a": left["total_amount"],
                "total_amount_b": right["total_amount"],
                "multipack_count_a": left["multipack_count"],
                "multipack_count_b": right["multipack_count"],
                "bi_encoder_score": round(float(score), 6),
                "faiss_rank": rank,
            }

faiss_pairs = pd.DataFrame(pair_rows.values()).sort_values(
    ["bi_encoder_score", "faiss_rank"], ascending=[False, True]
).reset_index(drop=True)
if faiss_pairs.empty:
    raise RuntimeError(
        "FAISS не нашёл соседей выше MY_FAISS_MIN_SCORE. Попробуйте снизить MY_FAISS_MIN_SCORE "
        "или увеличить MY_FAISS_TOP_K."
    )

print(f"FAISS pairs after score cutoff: {len(faiss_pairs):,}")
display(
    faiss_pairs[
        [
            "bi_encoder_score",
            "faiss_rank",
            "marketplace_a",
            "sku_a",
            "brand_a",
            "title_a",
            "marketplace_b",
            "sku_b",
            "brand_b",
            "title_b",
        ]
    ].head(MY_DISPLAY_ROWS)
)

## 8. Cross-encoder: пересчитываем score пары

Cross-encoder видит оба SKU одновременно, поэтому он дороже, но точнее для финального решения по edge. Если модель недоступна, demo явно покажет fallback на bi-encoder score.

In [ ]:
scored_pairs = faiss_pairs.head(MY_CROSS_ENCODER_PAIR_LIMIT).copy()
cross_runtime = _resolve_demo_cross_encoder(MY_CROSS_ENCODER_MODEL)
cross_threshold = (
    float(MY_CROSS_ENCODER_THRESHOLD)
    if MY_CROSS_ENCODER_THRESHOLD is not None
    else float(cross_runtime["threshold"] if cross_runtime["threshold"] is not None else 0.5)
)

cross_status_message = "skipped by MY_SKIP_CROSS_ENCODER"
cross_scores = [math.nan] * len(scored_pairs)
if not MY_SKIP_CROSS_ENCODER:
    cross_matcher = CrossEncoderMatcher(
        CrossEncoderConfig(
            model_name=str(cross_runtime["model_name"]),
            method_name=str(cross_runtime["method_name"]),
            batch_size=int(cross_runtime["batch_size"]),
            device=cross_runtime["device"],
            trust_remote_code=bool(cross_runtime["trust_remote_code"]),
            prompts=cross_runtime["prompts"],
            default_prompt_name=cross_runtime["default_prompt_name"],
            activation=cross_runtime["activation"],
        )
    )
    status = cross_matcher.status()
    cross_status_message = status.message
    if status.available:
        cross_scores = cross_matcher.score_batch(scored_pairs.to_dict("records"))
    elif MY_REQUIRE_CROSS_ENCODER:
        raise RuntimeError(f"Cross-encoder недоступен: {status.message}")

scored_pairs["cross_encoder_score"] = pd.to_numeric(pd.Series(cross_scores), errors="coerce")
has_cross_scores = scored_pairs["cross_encoder_score"].notna().any()
if MY_REQUIRE_CROSS_ENCODER and not has_cross_scores:
    raise RuntimeError(f"Cross-encoder не вернул score. Status: {cross_status_message}")

if has_cross_scores:
    scored_pairs["edge_score"] = scored_pairs["cross_encoder_score"]
    scored_pairs["score_source"] = str(cross_runtime["method_name"])
    scored_pairs["score_threshold"] = cross_threshold
    scored_pairs["predicted_label"] = np.where(
        scored_pairs["cross_encoder_score"].ge(cross_threshold),
        "exact_duplicate",
        "different_product",
    )
else:
    scored_pairs["edge_score"] = scored_pairs["bi_encoder_score"]
    scored_pairs["score_source"] = "bi_encoder_fallback"
    scored_pairs["score_threshold"] = MY_BI_ENCODER_THRESHOLD
    scored_pairs["predicted_label"] = np.where(
        scored_pairs["bi_encoder_score"].ge(MY_BI_ENCODER_THRESHOLD),
        "exact_duplicate",
        "different_product",
    )

print(f"Cross-encoder model: {cross_runtime['display_name']} [{cross_runtime['source_note']}] -> {cross_runtime['model_name']}")
print(f"Cross-encoder activation: {cross_runtime['activation'] or 'model default'}")
print(f"Cross-encoder status: {cross_status_message}")
print(f"Using score source: {scored_pairs['score_source'].iloc[0]}; threshold={scored_pairs['score_threshold'].iloc[0]}")
print(scored_pairs["predicted_label"].value_counts(dropna=False).to_string())

display(
    scored_pairs[
        [
            "predicted_label",
            "score_source",
            "edge_score",
            "bi_encoder_score",
            "cross_encoder_score",
            "marketplace_a",
            "sku_a",
            "brand_a",
            "title_a",
            "marketplace_b",
            "sku_b",
            "brand_b",
            "title_b",
        ]
    ].sort_values(["predicted_label", "edge_score"], ascending=[True, False]).head(MY_DISPLAY_ROWS)
)


## 9. Собираем SKU в группы

Строим граф: вершины — SKU, positive edges — пары `exact_duplicate`. Для demo по умолчанию используется connected components, потому что это самый простой способ показать механику группировки.


In [ ]:
group_input = scored_pairs.rename(columns={"edge_score": "score"}).copy()
components = build_graph_groups(
    group_input,
    edge_labels={"exact_duplicate"},
    config=ComponentConfig(
        left_id_col="raw_record_id_a",
        right_id_col="raw_record_id_b",
        label_col="predicted_label",
        component_col="demo_group_id",
    ),
    grouping_config=GraphGroupingConfig(
        algorithm=MY_GROUPING_ALGORITHM,
        edge_weight_col="score",
        default_edge_weight=1.0,
        seed=MY_RANDOM_STATE,
    ),
)

# Добавляем singleton SKU, если они были в demo-pool, но не попали ни в одну scored pair после лимитов.
missing_nodes = sorted(set(demo_records["raw_record_id"]) - set(components["node_id"]))
if missing_nodes:
    start = int(components["demo_group_id"].max()) if not components.empty else 0
    components = pd.concat(
        [
            components,
            pd.DataFrame(
                {"node_id": node_id, "demo_group_id": start + idx + 1}
                for idx, node_id in enumerate(missing_nodes)
            ),
        ],
        ignore_index=True,
    )

component_sizes = (
    components.groupby("demo_group_id", as_index=False)
    .agg(sku_count=("node_id", "nunique"))
    .sort_values(["sku_count", "demo_group_id"], ascending=[False, True])
)

print(f"Groups: {components['demo_group_id'].nunique():,}; multi-SKU groups: {int(component_sizes['sku_count'].gt(1).sum())}")
display(component_sizes.head(MY_DISPLAY_ROWS))

## 10. Join с кубом и выбор каноничного SKU

Для каждой группы считаем node-level edge stats. Каноничный SKU выбирается по среднему score positive edges, затем по продажам. Level 1 агрегирует продажи/выручку всей группы, Level 2 показывает реальные SKU внутри группы.

In [ ]:
node_edge_rows: list[dict[str, object]] = []
positive_edges = scored_pairs[scored_pairs["predicted_label"].eq("exact_duplicate")].copy()
for row in positive_edges.itertuples(index=False):
    for node_id in [row.raw_record_id_a, row.raw_record_id_b]:
        node_edge_rows.append(
            {
                "node_id": node_id,
                "positive_edge_score": float(row.edge_score),
                "positive_bi_encoder_score": float(row.bi_encoder_score),
                "positive_cross_encoder_score": float(row.cross_encoder_score) if pd.notna(row.cross_encoder_score) else math.nan,
            }
        )

if node_edge_rows:
    node_score_stats = (
        pd.DataFrame(node_edge_rows)
        .groupby("node_id", as_index=False)
        .agg(
            canonical_model_score=("positive_edge_score", "mean"),
            max_positive_edge_score=("positive_edge_score", "max"),
            positive_edge_count=("positive_edge_score", "size"),
        )
    )
else:
    node_score_stats = pd.DataFrame(columns=["node_id", "canonical_model_score", "max_positive_edge_score", "positive_edge_count"])

nodes = (
    components.merge(demo_records, left_on="node_id", right_on="raw_record_id", how="left", validate="one_to_one")
    .merge(sku_catalog.add_prefix("cube_"), left_on="node_id", right_on="cube_node_id", how="left", validate="one_to_one")
    .merge(node_score_stats, on="node_id", how="left")
)
nodes["canonical_model_score"] = nodes["canonical_model_score"].fillna(0.0)
nodes["positive_edge_count"] = nodes["positive_edge_count"].fillna(0).astype(int)
nodes["cube_sales_qty"] = pd.to_numeric(nodes["cube_sales_qty"], errors="coerce").fillna(0.0)
nodes["cube_revenue_rub"] = pd.to_numeric(nodes["cube_revenue_rub"], errors="coerce").fillna(0.0)

canonical_rows = []
for group_id, group in nodes.groupby("demo_group_id", sort=False):
    ordered = group.sort_values(
        ["canonical_model_score", "cube_sales_qty", "positive_edge_count", "node_id"],
        ascending=[False, False, False, True],
    )
    canonical = ordered.iloc[0]
    canonical_rows.append(
        {
            "demo_group_id": group_id,
            "canonical_node_id": canonical["node_id"],
            "canonical_marketplace": canonical.get("cube_Маркетплейс"),
            "canonical_sku": canonical.get("cube_Артикул"),
            "canonical_brand": canonical.get("cube_Бренд"),
            "canonical_title": canonical.get("cube_SKU"),
            "canonical_model_score": canonical["canonical_model_score"],
            "canonical_sales_qty": canonical["cube_sales_qty"],
        }
    )

canonical_df = pd.DataFrame(canonical_rows)
nodes = nodes.merge(canonical_df[["demo_group_id", "canonical_node_id"]], on="demo_group_id", how="left")

level1_rows = []
for group_id, group in nodes.groupby("demo_group_id", sort=False):
    canonical = canonical_df[canonical_df["demo_group_id"].eq(group_id)].iloc[0]
    level1_rows.append(
        {
            "level": "1_canonical_group",
            "demo_group_id": group_id,
            "canonical_node_id": canonical["canonical_node_id"],
            "node_id": canonical["canonical_node_id"],
            "marketplace": canonical["canonical_marketplace"],
            "sku": canonical["canonical_sku"],
            "brand": canonical["canonical_brand"],
            "title": canonical["canonical_title"],
            "sku_count_in_group": group["node_id"].nunique(),
            "group_sales_qty": round(float(group["cube_sales_qty"].sum()), 2),
            "group_revenue_rub": round(float(group["cube_revenue_rub"].sum()), 2),
            "group_avg_price_rub": round(float(group["cube_revenue_rub"].sum() / group["cube_sales_qty"].sum()), 2) if group["cube_sales_qty"].sum() else math.nan,
            "canonical_model_score": round(float(canonical["canonical_model_score"]), 6),
            "positive_edge_count": int(group["positive_edge_count"].sum()),
            "model_score_source": scored_pairs["score_source"].iloc[0],
        }
    )

level1 = pd.DataFrame(level1_rows)
level2 = nodes.assign(level="2_group_member").rename(
    columns={
        "cube_Маркетплейс": "marketplace",
        "cube_Артикул": "sku",
        "cube_Бренд": "brand",
        "cube_SKU": "title",
        "cube_sales_qty": "sales_qty",
        "cube_revenue_rub": "revenue_rub",
        "cube_avg_price_rub": "avg_price_rub",
        "cube_cube_rows": "cube_rows",
        "cube_cube_months": "cube_months",
    }
)

member_cols = [
    "level",
    "demo_group_id",
    "canonical_node_id",
    "node_id",
    "marketplace",
    "sku",
    "brand",
    "title",
    "cube_Вес, кг (ед.)",
    "cube_Вес, кг",
    "cube_multipack_count",
    "sales_qty",
    "revenue_rub",
    "avg_price_rub",
    "canonical_model_score",
    "max_positive_edge_score",
    "positive_edge_count",
    "cube_rows",
    "cube_months",
]
member_cols = [column for column in member_cols if column in level2.columns]
level2 = level2[member_cols].sort_values(
    ["demo_group_id", "canonical_model_score", "sales_qty", "node_id"],
    ascending=[True, False, False, True],
)

glued_group_ids = level1[level1["sku_count_in_group"].gt(1)]["demo_group_id"].tolist()
level1_glued = level1[level1["demo_group_id"].isin(glued_group_ids)].sort_values(
    ["sku_count_in_group", "group_sales_qty"], ascending=[False, False]
)

print("Level 1: каноничные SKU + агрегаты группы")
display(level1_glued.head(MY_DISPLAY_ROWS) if not level1_glued.empty else level1.head(MY_DISPLAY_ROWS))

print("Level 2: SKU внутри выбранных групп")
visible_group_ids = level1_glued["demo_group_id"].head(10).tolist() if not level1_glued.empty else level1["demo_group_id"].head(10).tolist()
display(level2[level2["demo_group_id"].isin(visible_group_ids)].head(MY_DISPLAY_ROWS * 4))

## 11. Дерево: Level 1 -> Level 2

Это компактная структура, которую потом можно превращать в UI: каноничная строка идёт первой, под ней реальные SKU группы.

In [ ]:
tree_rows: list[dict[str, object]] = []
ordered_group_ids = (
    level1.sort_values(["sku_count_in_group", "group_sales_qty"], ascending=[False, False])["demo_group_id"].tolist()
)
for group_id in ordered_group_ids:
    header = level1[level1["demo_group_id"].eq(group_id)].iloc[0].to_dict()
    tree_rows.append({**header, "tree_label": f"[L1] canonical: {header.get('title')}"})
    members = level2[level2["demo_group_id"].eq(group_id)].copy()
    members = members.sort_values(["canonical_model_score", "sales_qty", "node_id"], ascending=[False, False, True])
    for member in members.to_dict("records"):
        tree_rows.append({**member, "tree_label": f"  [L2] {member.get('marketplace')} / {member.get('sku')}: {member.get('title')}"})

tree_view = pd.DataFrame(tree_rows)
visible_tree = tree_view[tree_view["demo_group_id"].isin(visible_group_ids)].copy()

tree_cols = [
    "level",
    "demo_group_id",
    "tree_label",
    "canonical_node_id",
    "node_id",
    "brand",
    "sku_count_in_group",
    "group_sales_qty",
    "group_revenue_rub",
    "sales_qty",
    "revenue_rub",
    "canonical_model_score",
    "positive_edge_count",
]
tree_cols = [column for column in tree_cols if column in visible_tree.columns]
display(visible_tree[tree_cols].head(MY_DISPLAY_ROWS * 5))

## 12. Сохраняем demo-export

CSV нужен только как воспроизводимый артефакт для просмотра результата. Он не является production identity mapping.

In [ ]:
if MY_EXPORT_DEMO_CSV:
    RUN_PATHS.reports_dir.mkdir(parents=True, exist_ok=True)
    export_path = RUN_PATHS.grouped_sku_demo_path
    export_cols = [
        "level",
        "demo_group_id",
        "tree_label",
        "canonical_node_id",
        "node_id",
        "marketplace",
        "sku",
        "brand",
        "title",
        "sku_count_in_group",
        "group_sales_qty",
        "group_revenue_rub",
        "group_avg_price_rub",
        "sales_qty",
        "revenue_rub",
        "avg_price_rub",
        "canonical_model_score",
        "positive_edge_count",
        "model_score_source",
    ]
    export_cols = [column for column in export_cols if column in tree_view.columns]
    tree_view[export_cols].to_csv(export_path, index=False)
    print(f"Saved demo tree CSV: {export_path} ({len(tree_view):,} rows)")
else:
    print("MY_EXPORT_DEMO_CSV=False: CSV не сохранялся.")

## 13. Итоговые выводы и как пользоваться

1. В первой ячейке выберите `MY_CATEGORY_RUN` и при необходимости `MY_DUCKDB_PATH`.
2. Запустите notebook сверху вниз.
3. Сначала посмотрите лексические кандидаты: это стартовая группа похожих SKU из DuckDB.
4. Затем посмотрите FAISS-pairs и cross-encoder scores: это модельные edge-кандидаты.
5. Финальный результат — таблицы **Level 1** и **Level 2**, а также `tree_view`: каноничный SKU группы и реальные SKU под ним.
6. Для большого полного прогона используется production/research job; этот notebook оставлен как быстрая и наглядная витрина.
